In [1]:
# import libraries

# import pandas for manipulation
import pandas as pd

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# name export path
export_gen_path = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Alternative Formats of GOGET/"
export_qc_path = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/'
# import conversion factors
conversion_factors = pd.read_csv('/content/drive/MyDrive/GOGET q1 2026/GOGET oil & gas extraction sites conversion factors for QC  2026 01 30 1633 - Conversion factors.csv')

# make timestamp
from datetime import datetime
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

print('this runs qc output will all be timestamped: ' + timestamp)

this runs qc output will all be timestamped: 2026-02-27_16-23-36


In [7]:
# import recent production and reserve sheet
prod_reserves = pd.read_csv(export_gen_path + 'reserves-production-2026-02-27T160021.csv')

# pull in just reserves
reserves = prod_reserves.copy()
reserves = prod_reserves[prod_reserves['Type'] == 'reserves']

# print shape of reserves
print(reserves.shape)

(16723, 12)


In [9]:
import re

def extract_country(project_name):
    # Extract content from the last set of parentheses
    match = re.search(r'\(([^)]+)\)[^\(]*$', project_name)
    if match:
        content = match.group(1).strip()
        # If there's a comma, take the last part (assumed to be the country)
        if ',' in content:
            return content.split(',')[-1].strip()
        return content
    return None

# Apply the function to create the 'Country/Area' column in the reserves DataFrame
reserves['Country/Area'] = reserves['Project Name'].apply(extract_country)

print("Extracted countries and created 'Country/Area' column in 'reserves' dataframe.")
# You can uncomment the line below to check some of the extracted countries
# print(reserves['Country/Area'].unique())

Extracted countries and created 'Country/Area' column in 'reserves' dataframe.


/tmp/ipython-input-861/1749947649.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reserves['Country/Area'] = reserves['Project Name'].apply(extract_country)


In [12]:
# import rf sheet
rf_sheet = pd.read_csv(export_qc_path + 'recent_reserve_byfuel_converted_df 2026-01-30_22-36-42 - rf sheet.csv')

# filter recent_reserve_byfuel_converted_df for rows where reserveClassification is listed in rf_sheet "description"
in_place_recent = reserves[reserves['Reserve Classification'].isin(rf_sheet['description'])].copy()

# for in_place_recent, create country specifc id by joining Country/Area - fuelDesc - reserveClassification
in_place_recent['country_specific_id'] = in_place_recent['Country/Area'] + ' - ' + in_place_recent['Fuel Description'] + ' - ' + in_place_recent['Reserve Classification']

# matching on "Combo" in rf_sheet bring over Recovery Factor from rf_sheet to in_place_recent
in_place_recent = in_place_recent.merge(rf_sheet[['Combo', 'Recovery Factor']], left_on='country_specific_id', right_on='Combo', how='left')

# Clean and convert 'Recovery Factor' to numeric
in_place_recent['Recovery Factor'] = in_place_recent['Recovery Factor'].str.replace('%', '', regex=False).astype(float) / 100

# apply Recovery Factor to shared unit (million boe) value creating "estimated recoverable"
in_place_recent['estimated recoverable (RF applied to in-place)'] = in_place_recent['Quantity (converted)'] * in_place_recent['Recovery Factor']

# tag all rows where estimated recoverable (1P_2P, 3P_2P) > 0 as "converted from 1p to 2p, 3P to 2p"
in_place_recent['conversion type'] = in_place_recent.apply(lambda row: f"RF applied (Recovery Factor: {row['Recovery Factor']:.2f})" if row['estimated recoverable (RF applied to in-place)'] > 0 else 'not converted', axis=1)

# rename estimated recoverable (RF applied to in-place)
in_place_recent['estimated recoverable'] = in_place_recent['estimated recoverable (RF applied to in-place)']

# drop columns
in_place_recent = in_place_recent.drop(columns=['Combo', 'country_specific_id', 'Recovery Factor','estimated recoverable (RF applied to in-place)'])

in_place_recent

,Project ID,Project Name,Type,Fuel Description,Reserve Classification,Quantity,Quantity Unit,Quantity (converted),Quantity Unit (converted),Year,Data Source,Notes,Country/Area,conversion type,estimated recoverable
0,L100000304986,AAP-ON-94/1 Oil and Gas Block (India),reserves,gas,original gas in place,2.440000e+02,bcf,6909.348,million m³,2019.0,https://www.hoec.com/wp-content/uploads/2019/0...,NaN,India,RF applied (Recovery Factor: 0.85),5872.94580
1,L100000300002,"Abalone Oil and Gas Field (Espírito Santo, Bra...",reserves,gas,original volume in place,3.137000e+03,million m³,3137.000,million m³,2015.0,http://www.anp.gov.br/images/planos_desenvolvi...,NaN,Brazil,RF applied (Recovery Factor: 0.85),2666.45000
2,L100000300002,"Abalone Oil and Gas Field (Espírito Santo, Bra...",reserves,oil,volume in place,3.356000e+01,million bbl,33.560,million bbl,2015.0,http://www.anp.gov.br/images/planos_desenvolvi...,NaN,Brazil,RF applied (Recovery Factor: 0.30),10.06800
3,L100000300002,"Abalone Oil and Gas Field (Espírito Santo, Bra...",reserves,gas,volume in place,1.732809e+09,m³,1732.809,million m³,2024.0,https://www.gov.br/anp/pt-br/centrais-de-conte...,NaN,Brazil,RF applied (Recovery Factor: 0.85),1472.88765
4,L100000300002,"Abalone Oil and Gas Field (Espírito Santo, Bra...",reserves,oil,volume in place,1.500120e+07,bbl,15.001,million bbl,2024.0,https://www.gov.br/anp/pt-br/centrais-de-conte...,NaN,Brazil,RF applied (Recovery Factor: 0.30),4.50030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2478,L100000312598,Zhetybay Oil Field (Kazakhstan),reserves,oil,geological reserves,3.300000e+02,million tons,2194.401,million bbl,2017.0,https://lsm.kz/%20neftyanye-mestorozhdeniya,"According to the source: ""the estimated reserv...",Kazakhstan,RF applied (Recovery Factor: 0.30),658.32030
2479,L100000301670,Zielin Oil and Gas Field (Poland),reserves,oil and condensate,economic resources in place,2.330000e+00,thousand tonnes,0.017,million bbl,2019.0,http://geoportal.pgi.gov.pl/css/surowce/images...,NaN,Poland,RF applied (Recovery Factor: 0.30),0.00510
2480,L100000311578,Zohr Gas Field (Egypt),reserves,gas,initial resource in place,3.000000e+01,tcf,849510.000,million m³,2016.0,https://www.egypttoday.com/Article/3/131750/Eg...,NaN,Egypt,RF applied (Recovery Factor: 0.85),722083.50000
2481,L100000305210,Zorro Oil Field (Venezuela),reserves,oil,oil in place,5.380000e+02,million bbl,538.000,million bbl,2021.0,https://eprinc.org/wp-content/uploads/2021/09/...,NaN,Venezuela,RF applied (Recovery Factor: 0.13),69.94000


In [20]:
# import 1P2P3P sheet
p1p2p3p_sheet = pd.read_csv(export_qc_path + 'recent_reserve_byfuel_converted_df 2026-01-30_22-36-42 - Clean 2p conversion.csv')

# filter recent_reserve_byfuel_converted_df for rows where reserveClassification is listed in p1p2p3p "reserveClassification"
closeto1p2p3p = reserves[reserves['Reserve Classification'].isin(p1p2p3p_sheet['reserveClassification'])].copy()

# map p1p2p3p_sheet Conversion_factor to closeto1p2p3p on reserveClassification
closeto1p2p3p = closeto1p2p3p.merge(p1p2p3p_sheet[['reserveClassification', 'Conversion_factor']], left_on='Reserve Classification', right_on='reserveClassification', how='left')

# apply Recovery Factor to shared unit (million boe) value creating "estimated recoverable"
closeto1p2p3p['estimated recoverable (1P_2P, 3P_2P)'] = closeto1p2p3p['Quantity (converted)'] * closeto1p2p3p['Conversion_factor']

# tag all rows where estimated recoverable (1P_2P, 3P_2P) > 0 as "converted from 1p to 2p, 3P to 2p"
closeto1p2p3p['conversion type'] = closeto1p2p3p.apply(lambda row: f"converted from 1p to 2p, 3P to 2p (Conversion Factor: {row['Conversion_factor']:.2f})"  if row['estimated recoverable (1P_2P, 3P_2P)'] > 0 else 'not converted', axis=1)

# rename estimated recoverable (RF applied to in-place)
closeto1p2p3p['estimated recoverable'] = closeto1p2p3p['estimated recoverable (1P_2P, 3P_2P)']

# drop columns
closeto1p2p3p = closeto1p2p3p.drop(columns=['Conversion_factor', 'estimated recoverable (1P_2P, 3P_2P)'])

closeto1p2p3p

,Project ID,Project Name,Type,Fuel Description,Reserve Classification,Quantity,Quantity Unit,Quantity (converted),Quantity Unit (converted),Year,Data Source,Notes,Country/Area,reserveClassification,conversion type,estimated recoverable
0,L100000318123,16/1-34 S (Lillefix) Oil and Gas Field (Norway),reserves,gas,original recoverable reserves,0.09,billion Sm³,90.000,million m³,2021.0,https://factpages.npd.no/factpages/default.asp...,NaN,Norway,original recoverable reserves,"converted from 1p to 2p, 3P to 2p (Conversion ...",90.000
1,L100000318123,16/1-34 S (Lillefix) Oil and Gas Field (Norway),reserves,NGL,original recoverable reserves,0.03,million Sm³,0.189,million bbl,2021.0,https://factpages.npd.no/factpages/default.asp...,NaN,Norway,original recoverable reserves,"converted from 1p to 2p, 3P to 2p (Conversion ...",0.189
2,L100000318123,16/1-34 S (Lillefix) Oil and Gas Field (Norway),reserves,oil,original recoverable reserves,0.86,million Sm³,5.409,million bbl,2021.0,https://factpages.npd.no/factpages/default.asp...,NaN,Norway,original recoverable reserves,"converted from 1p to 2p, 3P to 2p (Conversion ...",5.409
3,L100000318123,16/1-34 S (Lillefix) Oil and Gas Field (Norway),reserves,condensate,7F,0.00,million Sm³,0.000,million bbl,2023.0,https://factpages.sodir.no/en/discovery/TableV...,NaN,Norway,7F,not converted,0.000
4,L100000318123,16/1-34 S (Lillefix) Oil and Gas Field (Norway),reserves,gas,7F,0.09,billion Sm³,90.000,million m³,2023.0,https://factpages.sodir.no/en/discovery/TableV...,NaN,Norway,7F,"converted from 1p to 2p, 3P to 2p (Conversion ...",90.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14171,L100000314116,Zuluf Oil and Gas Field (Saudi Arabia),reserves,gas,remaining reserves,1766.64,billion scf,50025.945,million m³,2019.0,https://www.offshore-technology.com/features/w...,NaN,Saudi Arabia,remaining reserves,"converted from 1p to 2p, 3P to 2p (Conversion ...",50025.945
14172,L100000314116,Zuluf Oil and Gas Field (Saudi Arabia),reserves,oil,remaining reserves,11072.35,million bbl,11072.350,million bbl,2019.0,https://www.offshore-technology.com/features/w...,NaN,Saudi Arabia,remaining reserves,"converted from 1p to 2p, 3P to 2p (Conversion ...",11072.350
14173,L100000321174,Zumlat al-Maharr Gas Field (Syria),reserves,gas,estimated,9.00,bcm,9000.000,million m³,2022.0,https://english.enabbaladi.net/archives/2022/0...,NaN,Syria,estimated,"converted from 1p to 2p, 3P to 2p (Conversion ...",9000.000
14174,L100000305209,Zumo Oil Field (Venezuela),reserves,oil,EUR,118.00,million bbl,118.000,million bbl,2000.0,https://pubs.usgs.gov/cp/50/report.pdf,NaN,Venezuela,EUR,"converted from 1p to 2p, 3P to 2p (Conversion ...",118.000


In [28]:
# join closeto1p2p3p and in_place_recent
in_place_and1p2p3p = pd.concat([in_place_recent, closeto1p2p3p])

# labelling columns to keep
keep_columns = ('Project ID', 'Project Name', 'Country/Area', 'Type', 'Fuel Description',
       'Reserve Classification', 'Quantity', 'Quantity Unit',
        'Quantity (converted)', 'Quantity Unit (converted)', 'Year','conversion type', 'estimated recoverable')
# drop non-keep columns, reorder to match keep list
in_place_and1p2p3p = in_place_and1p2p3p[list(keep_columns)]

In [29]:
in_place_and1p2p3p.columns

Index(['Project ID', 'Project Name', 'Country/Area', 'Type',
       'Fuel Description', 'Reserve Classification', 'Quantity',
       'Quantity Unit', 'Quantity (converted)', 'Quantity Unit (converted)',
       'Year', 'conversion type', 'estimated recoverable'],
      dtype='object')

In [72]:
# rename
harmonized_reserves = in_place_and1p2p3p.copy()

# pull in just production
all_prod = prod_reserves.copy()
all_prod = prod_reserves[prod_reserves['Type'].isin(['production', 'production design capacity'])]

In [62]:
# define million m³/y to million boe/y conversion
mcm_mboe = 0.00588

In [73]:
# when "Common units" = "million m³/y" multiply by .00588 new "shared unit" column
all_prod.loc[all_prod['Quantity Unit (converted)'] == 'million m³/y', 'shared unit (million boe/y)'] = all_prod['Quantity (converted)'] * .00588

# when common units = million bbl/y or million boe/y, put that value in shared unit column
all_prod.loc[all_prod['Quantity Unit (converted)'].isin(['million bbl/y', 'shared unit (million boe/y)']), 'shared unit (million boe/y)'] = all_prod['Quantity (converted)']
all_prod.loc[all_prod['Quantity Unit (converted)'].isin(['million boe/y', 'shared unit (million boe/y)']), 'shared unit (million boe/y)'] = all_prod['Quantity (converted)']

# sort recent_production_byfuel_converted_df by shared unit (million boe/y) in ascending order
all_prod = all_prod.sort_values(by=['shared unit (million boe/y)'], ascending=[False])




/tmp/ipython-input-861/795316113.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_prod.loc[all_prod['Quantity Unit (converted)'] == 'million m³/y', 'shared unit (million boe/y)'] = all_prod['Quantity (converted)'] * .00588


In [81]:
# Define common identifying columns for comparison
# Note: 'estimated recoverable' and 'conversion type' are only in harmonized_reserves, so they are excluded from the merge key
common_cols = [
    'Project ID', 'Project Name', 'Type', 'Fuel Description',
    'Reserve Classification', 'Quantity', 'Quantity Unit',
    'Quantity (converted)', 'Quantity Unit (converted)', 'Year', 'Country/Area'
]

# Perform a left merge to find rows in reserves that are not in harmonized_reserves
# using indicator=True to show merge origins
reserves_not_in_harmonized = reserves.merge(
    harmonized_reserves[common_cols], # Merge only on common columns from harmonized_reserves
    on=common_cols,
    how='left',
    indicator=True
)

# Filter for rows that are only present in the 'left' (reserves) DataFrame
reserves_only = reserves_not_in_harmonized[reserves_not_in_harmonized['_merge'] == 'left_only'].drop(columns='_merge')

print(f"Found {len(reserves_only)} entries in 'reserves' that are not in 'harmonized_reserves'.")

if not reserves_only.empty:
    print("Here are the first 5 rows of reserves not in harmonized_reserves:")
    print(reserves_only.head())
else:
    print("All entries in 'reserves' appear to be included in 'harmonized_reserves'.")

Found 68 entries in 'reserves' that are not in 'harmonized_reserves'.
Here are the first 5 rows of reserves not in harmonized_reserves:
        Project ID                                    Project Name      Type  \
56   L100000320770  35/10-10 S (Carmen) Oil and Gas Field (Norway)  reserves   
497  L100000312689       Akshabulak Oil and Gas Field (Kazakhstan)  reserves   
565  L100000312691       Alibekmola Oil and Gas Field (Kazakhstan)  reserves   
584  L100000321163            Al Khairat Oil and Gas Field (Syria)  reserves   
894  L100000310920           Aripo Gas Field (Trinidad and Tobago)  reserves   

    Fuel Description        Reserve Classification  Quantity Quantity Unit  \
56               oil                            2F      2.49   million Sm³   
497              oil                  [not stated]    323.00   million bbl   
565     [not stated]                  [not stated]    250.00   million bbl   
584     hydrocarbons                            3C     73.00   million 

In [65]:
reserves_only.drop(columns=['Notes', 'Data Source'], inplace=True)

In [66]:
# append harmonized_reserves and reserves_not_in_harmonized as "all reserves"
all_reserves = pd.concat([harmonized_reserves, reserves_only])

In [74]:
all_prod = all_prod.drop(columns=['Notes', 'Reserve Classification'])

In [77]:
# drop index
all_prod = all_prod.reset_index(drop=True)
all_reserves = all_reserves.reset_index(drop=True)

In [78]:
# import excel writer
from pandas import ExcelWriter
import os

# Ensure the export directory exists
os.makedirs(export_gen_path, exist_ok=True)

# save all_prod and harmonized_reserves to a excel titled "Full Production and Reserves"
with ExcelWriter(export_gen_path + f'Full Production and Reserves {timestamp}.xlsx') as writer:
    all_prod.to_excel(writer, sheet_name='Production')
    all_reserves.to_excel(writer, sheet_name='Reserves')
